# UTUEL — Evaluation Notebook for Prompt Pipeline

Compile all pipeline output files into a single DataFrame, normalise ground-truth
and prediction values, then compute per-model accuracy metrics.

## 1 · Import Required Libraries

In [1]:
import subprocess
import sys
import json
from pathlib import Path

import pandas as pd

# Add project root to path so the package is importable from the notebook
PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from compile import (
    compile_all,
    compile_dataset,
    normalize_answer,
    is_correct,
)

DATASETS_DIR = PROJECT_ROOT / "datasets" /"results"
print(f"Project root : {PROJECT_ROOT}")
print(f"Datasets dir : {DATASETS_DIR}")

Project root : C:\Users\wtchuitc\Documents\GitHub\UTUEL
Datasets dir : C:\Users\wtchuitc\Documents\GitHub\UTUEL\datasets\results


## 2 · Compile Pipeline Outputs

Scan every `datasets/<name>/<model>/run<N>.jsonl` and merge into
`datasets/compiled_<name>/compiled.jsonl`.  
Already-compiled files are overwritten so results are always fresh.

In [2]:
# ── debug: show source run files, excluding compiled_* output folders ─────────
print(f"DATASETS_DIR exists : {DATASETS_DIR.exists()}")
print(f"DATASETS_DIR        : {DATASETS_DIR}\n")

# compile_all() discovers every dataset folder recursively and writes
# compiled_<name>/compiled.jsonl for each one, overwriting any previous run.
compiled_paths = compile_all(DATASETS_DIR)

print("\nCompiled files:")
if compiled_paths:
    for p in compiled_paths:
        print(f"  {p.relative_to(PROJECT_ROOT)}")
else:
    print("  (none — no *.jsonl files found yet)")

DATASETS_DIR exists : True
DATASETS_DIR        : C:\Users\wtchuitc\Documents\GitHub\UTUEL\datasets\results

[compile] test_lookup_WikiSQL → C:\Users\wtchuitc\Documents\GitHub\UTUEL\datasets\results\compiled_test_lookup_WikiSQL\compiled.jsonl  (90592 records)

Compiled files:
  datasets\results\compiled_test_lookup_WikiSQL\compiled.jsonl


## 3 · Load Compiled JSONL into a DataFrame

Load all compiled files and concatenate them into one DataFrame.

In [3]:
def load_compiled_jsonl(path: Path) -> pd.DataFrame:
    """Read a compiled JSONL file into a DataFrame."""
    records = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


# Discover all compiled.jsonl files under datasets/compiled_*/
# This works even if the compile cell above was not re-run this session.
compiled_files = sorted(DATASETS_DIR.glob("compiled_*/compiled.jsonl"))

if not compiled_files:
    raise RuntimeError(
        "No compiled.jsonl files found under datasets/compiled_*/\n"
        "Run the compile cell above first."
    )

print("Loading:")
frames = []
for p in compiled_files:
    frame = load_compiled_jsonl(p)
    print(f"  {p.relative_to(DATASETS_DIR)}  ({len(frame)} records)")
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

print(f"\nTotal records : {len(df)}")
print(f"Models        : {sorted(df['model'].unique())}")
print(f"Columns       : {list(df.columns)}")
df.head(3)

Loading:
  compiled_test_lookup_WikiSQL\compiled.jsonl  (90592 records)

Total records : 90592
Models        : ['TableGPT2-7B', 'deepseek-r1', 'gemma2', 'gemma4', 'gpt-oss', 'llama3', 'qwen3.6:27b', 'tablellm-7b-Q4']
Columns       : ['table_id', 'ground_truth', 'question', 'prompt', 'model', 'run', 'response', 'prediction', 'parse_ok', 'correct', 'token_in_prompt', 'token_in_output']


,table_id,ground_truth,question,prompt,model,run,response,prediction,parse_ok,correct,token_in_prompt,token_in_output
0,2-14642287-5,[1],How many bronze medals are there when there ar...,"Task Description: Please look at the table, an...",deepseek-r1,1,"{""answer"": ""1""}",1,True,True,86,2
1,2-15847138-2,"[Colonial Square, Masters]",Which Event has a 2007–08 of n/a?,"Task Description: Please look at the table, an...",deepseek-r1,1,"{""answer"": ""Colonial Square and Masters""}",Colonial Square and Masters,True,False,78,5
2,2-13771649-13,[32 - 08],What is the points difference associated with ...,"Task Description: Please look at the table, an...",deepseek-r1,1,"{""answer"": ""32 - 08""}",32 - 08,True,True,95,4


## 4 · Normalise Ground-Truth Values

`ground_truth` may be a `list[str]` (most rows) or a plain `str`.  
`normalize_answer()` from `compile.py` lower-cases, collapses whitespace, and
joins lists with `", "` so both sides of a comparison are in the same shape.

## 5 · Per-Model Accuracy Aggregation

Aggregate by `model` (and optionally `run`) to compute:
- **accuracy** — fraction of rows where `correct == True`
- **parse_rate** — fraction of rows where the JSON answer was successfully extracted
- **n** — total number of evaluated rows

In [7]:
def format_compact_number(value):
    value = float(value)
    abs_value = abs(value)
    if abs_value >= 1_000_000_000:
        return f"{value / 1_000_000_000:.2f}B"
    if abs_value >= 1_000_000:
        return f"{value / 1_000_000:.2f}M"
    if abs_value >= 1_000:
        return f"{value / 1_000:.2f}K"
    return f"{value:.2f}"


df_eval = df.assign(
    token_in_prompt=df["token_in_prompt"].fillna(0),
    token_in_output=df["token_in_output"].fillna(0),
)
df_eval["total_tokens"] = df_eval["token_in_prompt"] + df_eval["token_in_output"]

summary = (
    df_eval.groupby(["model", "run"])
.agg(
        n               =("correct", "count"),
        accuracy        =("correct", "mean"),
        parse_rate      =("parse_ok", "mean"),
        # token_in_prompt =("token_in_prompt", "sum"),
        # token_in_output =("token_in_output", "sum"),
        total_tokens    =("total_tokens", "sum"),
    )
.reset_index()
.sort_values(["model", "run"])
)

summary["accuracy"] = summary["accuracy"].map("{:.1%}".format)
summary["parse_rate"] = summary["parse_rate"].map("{:.1%}".format)
for column in ["total_tokens"]:
    summary[column] = summary[column].map(format_compact_number)

summary

,model,run,n,accuracy,parse_rate,total_tokens
0,TableGPT2-7B,1,11324,75.9%,99.8%,2.11M
1,deepseek-r1,1,11324,68.8%,99.5%,2.11M
2,gemma2,1,11324,69.5%,99.8%,2.56M
3,gemma4,1,11324,83.2%,100.0%,2.27M
4,gpt-oss,1,11324,79.6%,100.0%,2.10M
5,llama3,1,11324,56.8%,96.7%,2.84M
6,qwen3.6:27b,1,11324,88.0%,100.0%,2.54M
7,tablellm-7b-Q4,1,11324,22.9%,61.5%,2.27M


### Overall accuracy collapsed across runs

In [ ]:
overall = (
    df_eval.groupby("model")
.agg(
        n               =("correct", "count"),
        accuracy        =("correct", "mean"),
        parse_rate      =("parse_ok", "mean"),
        token_in_prompt =("token_in_prompt", "sum"),
        token_in_output =("token_in_output", "sum"),
        total_tokens    =("total_tokens", "sum"),
    )
.reset_index()
.sort_values("accuracy", ascending=False)
)

overall["accuracy"] = overall["accuracy"].map("{:.1%}".format)
overall["parse_rate"] = overall["parse_rate"].map("{:.1%}".format)
for column in ["token_in_prompt", "token_in_output", "total_tokens"]:
    overall[column] = overall[column].map(format_compact_number)

overall

,model,n,accuracy,parse_rate
6,qwen3.6:27b,11324,88.0%,100.0%
3,gemma4,11324,83.2%,100.0%
4,gpt-oss,11324,79.6%,100.0%
0,TableGPT2-7B,11324,75.9%,99.8%
2,gemma2,11324,69.5%,99.8%
1,deepseek-r1,11324,68.8%,99.5%
5,llama3,11324,56.8%,96.7%
7,tablellm-7b-Q4,11324,22.9%,61.5%


In [9]:
path = "D:\\TABLE_DATASET\\WIKI_TABLE_SQL\\test_lookup.jsonl"

In [10]:
import json
from pathlib import Path
import pandas as pd

src_path = Path(path)
if not src_path.exists():
    raise FileNotFoundError(f"JSONL file not found: {src_path}")

rows = []
with src_path.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue

        rows.append({
            "id": obj.get("id"),
            "table_id": obj.get("table_id"),
            "question": obj.get("question"),
            "header": obj.get("header"),
        })

extracted_df = pd.DataFrame(rows, columns=["id", "table_id", "question", "header"])

out_path = src_path.with_name(src_path.stem + "_id_table_question_header.jsonl")
with out_path.open("w", encoding="utf-8") as out_f:
    for rec in extracted_df.to_dict(orient="records"):
        out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Loaded rows : {len(extracted_df)}")
print(f"Saved file  : {out_path}")
extracted_df.head(10)

Loaded rows : 11324
Saved file  : D:\TABLE_DATASET\WIKI_TABLE_SQL\test_lookup_id_table_question_header.jsonl


,id,table_id,question,header
0,0,1-10015132-16,What is terrence ross' nationality,"[Player, No., Nationality, Position, Years in ..."
1,1,1-10015132-16,What clu was in toronto 1995-96,"[Player, No., Nationality, Position, Years in ..."
2,2,1-10015132-16,which club was in toronto 2003-06,"[Player, No., Nationality, Position, Years in ..."
3,4,1-10083598-1,Where was Assen held?,"[No, Date, Round, Circuit, Pole Position, Fast..."
4,6,1-10083598-1,What was the date of the race in Misano?,"[No, Date, Round, Circuit, Pole Position, Fast..."
5,8,1-1013129-2,What are the nationalities of the player picke...,"[Pick, Player, Position, Nationality, NHL team..."
6,11,1-1013129-3,What's Dorain Anneck's pick number?,"[Pick, Player, Position, Nationality, NHL team..."
7,12,1-1013129-3,What is the nationality of the player from Van...,"[Pick, Player, Position, Nationality, NHL team..."
8,13,1-1013129-3,What's the pick number of the player from Spri...,"[Pick, Player, Position, Nationality, NHL team..."
9,14,1-1014206-2,When were the ships launched that were laid do...,"[#, Shipyard, Laid down, Launched, Commissione..."
